In [6]:
# %%
import os
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import distance_transform_edt, binary_closing, label
from skimage.morphology import skeletonize

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================

# Path updated directly to the refined_masks directory
IMAGE_FOLDER = r"C:\Users\ishin\OneDrive\Desktop\Ishini\images\refined_masks" 
OUTPUT_CSV = os.path.join(IMAGE_FOLDER, "gbm_thickness_results.csv")
OUTPUT_PLOT = os.path.join(IMAGE_FOLDER, "global_gbm_distribution.png")

# Scale factor from TEM scale bar
NM_PER_PIXEL = 0.9901  

# Physical validation bounds in nm
MIN_VALID_NM = 10.0   
MAX_VALID_NM = 1500.0 

# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def classify_thickness(mean_thickness):
    """Classifies mean GBM thickness based on biological standards (in nm)."""
    if mean_thickness < 200.0:
        return "Thin"
    elif 200.0 <= mean_thickness <= 450.0:
        return "Normal"
    else:
        return "Thick"

def process_single_image(image_path):
    """Processes a standard binary mask where white (255) = membrane."""
    filename = os.path.basename(image_path)
    
    # Parsing filenames like "01-24_03_1g.png"
    parts = filename.split('_')
    patient_id = parts[0] if len(parts) > 0 else "Unknown"
    view_id = parts[1] if len(parts) > 1 else "Unknown"
    glomerulus_id = parts[2].split('.')[0] if len(parts) > 2 else "Unknown"
    
    # Load grayscale mask
    img = Image.open(image_path).convert('L')
    img_array = np.array(img)
    
    # STANDARD THRESHOLDING FOR REFINED MASKS: White foreground > 128
    binary_mask = img_array > 128  
    
    if not np.any(binary_mask):
        return []

    # Morphological cleanup
    cleaned_mask = binary_closing(binary_mask, structure=np.ones((3, 3)))
    
    # Label connected membrane components
    labeled_mask, num_features = label(cleaned_mask)
    
    # Distance transform (distance to nearest background pixel)
    dist_trans = distance_transform_edt(cleaned_mask)
    
    # Skeletonization (medial axis)
    skeleton = skeletonize(cleaned_mask)
    
    # Calculate orthogonal thickness (2 * distance * scale)
    raw_thicknesses = dist_trans[skeleton] * 2.0 * NM_PER_PIXEL
    
    # Filter valid physical range
    valid_thicknesses = raw_thicknesses[
        (raw_thicknesses >= MIN_VALID_NM) & (raw_thicknesses <= MAX_VALID_NM)
    ]
    
    if len(valid_thicknesses) == 0:
        return []
        
    mean_thick = np.mean(valid_thicknesses)
    median_thick = np.median(valid_thicknesses)
    std_thick = np.std(valid_thicknesses)
    category = classify_thickness(mean_thick)
    
    return [{
        "Filename": filename,
        "Patient_ID": patient_id,
        "View_ID": view_id,
        "Glomerulus": glomerulus_id,
        "Num_Slices_Found": num_features,
        "Mean_Thickness_nm": mean_thick,
        "Median_Thickness_nm": median_thick,
        "Std_Dev_nm": std_thick,
        "Category": category,
        "Raw_Values": valid_thicknesses
    }]

# ==========================================
# 3. MAIN EXECUTION PIPELINE
# ==========================================
def main():
    print(f"Starting GBM Thickness Analysis on: {IMAGE_FOLDER}")
    
    # Look for PNGs inside the refined_masks folder
    image_paths = glob.glob(os.path.join(IMAGE_FOLDER, "*.png"))

    if not image_paths:
        # Fallback to TIF if the masks happen to be saved as binary TIFs
        image_paths = glob.glob(os.path.join(IMAGE_FOLDER, "*.[tT][iI][fF]*"))

    if not image_paths:
        print(f"No mask files found in {IMAGE_FOLDER}!")
        return

    print(f"Found {len(image_paths)} mask image(s).")
    
    all_results = []
    all_raw_thicknesses = []

    for path in image_paths:
        results = process_single_image(path)
        if results:
            all_results.extend(results)
            all_raw_thicknesses.extend(results[0]["Raw_Values"])
            print(f" Processed: {results[0]['Filename']} | Mean: {results[0]['Mean_Thickness_nm']:.2f} nm")

    if not all_results:
        print("No valid mask data processed. Please check mask directory contents.")
        return

    # Save summary dataframe
    df = pd.DataFrame(all_results)
    df_csv = df.drop(columns=["Raw_Values"]) 
    df_csv.to_csv(OUTPUT_CSV, index=False)
    print(f"\n Analysis results saved to:\n   {OUTPUT_CSV}")

    # Plot histogram
    plt.figure(figsize=(7, 4.5))
    plt.hist(all_raw_thicknesses, bins=80, density=True, color='mediumslateblue', alpha=0.7)
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.title("Global GBM Thickness Distribution", fontsize=12, fontweight='bold')
    plt.xlabel("Thickness (nm)", fontsize=10)
    plt.ylabel("Density", fontsize=10)
    plt.tight_layout()
    
    plt.savefig(OUTPUT_PLOT, dpi=300)
    plt.show()

if __name__ == "__main__":
    main()

Starting GBM Thickness Analysis on: C:\Users\ishin\OneDrive\Desktop\Ishini\images\refined_masks
No mask files found in C:\Users\ishin\OneDrive\Desktop\Ishini\images\refined_masks!
